# Methodology Review Diagnostics

This notebook focuses on the methodology-review questions for the MFA workshop paper. It starts from the dataset-level handoff matrices exported by `01_mfa.ipynb` and checks the load-bearing claims:

1. the surviving foundation-model association after confound adjustment,
2. the low-sample, high-p regime through multiplicity and effective rank,
3. simple leave-one-dataset-out baselines beyond the mean predictor.


## Setup

The notebook uses cached handoff tables when available. Re-run `01_mfa.ipynb` for `config_0.yaml` and `config_1.yaml` first if these files are missing.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

cwd = Path.cwd()
project_dir = cwd if (cwd / "src" / "mfa").exists() else cwd.parent
handoff_dir = project_dir / ".mfa_cache" / "notebook_handoff"

CONFIG_HANDOFFS = {
    "nn_vs_tree": "config_0_6d4d59e15d8c06f5",
    "standard_vs_foundational": "config_1_c4472762a9293af4",
}

CONTEXT_COLUMNS = {
    "dataset", "task_type", "comparison_name", "group_a_name", "group_b_name",
    "group_a_label", "group_b_label", "expected_direction", "n_splits",
    "best_a_error", "best_a_norm_error", "best_b_error", "best_b_norm_error",
    "delta_raw", "delta_raw_std", "delta_raw_sem", "delta_norm", "delta_norm_std", "delta_norm_sem",
}

RANDOM_SEED = 20260424
TARGET = "delta_norm"
MIN_N = 30


In [ ]:
def load_matrix(comparison: str, matrix: str) -> pd.DataFrame:
    path = handoff_dir / CONFIG_HANDOFFS[comparison] / f"analysis_{matrix}_reduced.pkl"
    if not path.exists():
        raise FileNotFoundError(f"Missing handoff matrix: {path}")
    return pd.read_pickle(path)


def numeric_feature_columns(frame: pd.DataFrame, *, min_n: int = MIN_N) -> list[str]:
    columns = []
    for column in frame.columns:
        if column in CONTEXT_COLUMNS or column.startswith("best_"):
            continue
        values = pd.to_numeric(frame[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if values.notna().sum() >= min_n and values.nunique(dropna=True) > 1:
            columns.append(column)
    return columns


def association_table(frame: pd.DataFrame) -> pd.DataFrame:
    y = pd.to_numeric(frame[TARGET], errors="coerce")
    rows = []
    for feature in numeric_feature_columns(frame):
        x = pd.to_numeric(frame[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
        valid = x.notna() & y.notna()
        result = stats.spearmanr(x[valid], y[valid])
        rows.append(
            {
                "feature": feature,
                "n": int(valid.sum()),
                "spearman_rho": float(result.statistic),
                "abs_spearman_rho": abs(float(result.statistic)),
                "p_value": float(result.pvalue),
            }
        )
    table = pd.DataFrame(rows)
    table["bh_q_value"] = multipletests(table["p_value"], alpha=0.05, method="fdr_bh")[1]
    return table.sort_values(["abs_spearman_rho", "feature"], ascending=[False, True]).reset_index(drop=True)


def bootstrap_spearman(frame: pd.DataFrame, feature: str, *, repeats: int = 2000) -> dict[str, float | int]:
    x = pd.to_numeric(frame[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
    y = pd.to_numeric(frame[TARGET], errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = x.notna() & y.notna()
    x_values = x[valid].to_numpy(dtype=float)
    y_values = y[valid].to_numpy(dtype=float)
    observed = float(stats.spearmanr(x_values, y_values).statistic)
    rng = np.random.default_rng(RANDOM_SEED)
    values = []
    for _ in range(repeats):
        idx = rng.integers(0, len(x_values), len(x_values))
        statistic = stats.spearmanr(x_values[idx], y_values[idx]).statistic
        if np.isfinite(statistic):
            values.append(float(statistic))
    values = np.asarray(values, dtype=float)
    return {
        "bootstrap_ci_low": float(np.quantile(values, 0.025)),
        "bootstrap_ci_high": float(np.quantile(values, 0.975)),
        "bootstrap_sign_consistency": float(np.mean(np.sign(values) == np.sign(observed))),
        "bootstrap_repeats": int(len(values)),
    }


## Association Screen Recap

This reproduces the top univariate signals and shows which, if any, pass BH at 0.05 before the bootstrap sign-consistency filter.

In [ ]:
association_summaries = []
robust_candidates = []

for comparison in CONFIG_HANDOFFS:
    for matrix in ["general", "classification"]:
        frame = load_matrix(comparison, matrix)
        assoc = association_table(frame)
        top = assoc.head(5).copy()
        top.insert(0, "matrix", matrix)
        top.insert(0, "comparison", comparison)
        association_summaries.append(top)

        for _, row in assoc[assoc["bh_q_value"].lt(0.05)].iterrows():
            boot = bootstrap_spearman(frame, row["feature"])
            robust_candidates.append({"comparison": comparison, "matrix": matrix, **row.to_dict(), **boot})

association_summary = pd.concat(association_summaries, ignore_index=True)
robust_table = pd.DataFrame(robust_candidates)

display(association_summary)
display(robust_table)


## Confound Sensitivity for `pymfe__attr_ent.skewness`

The review concern is that the one surviving feature could be a shadow of dataset size, dimensionality, categorical composition, or missingness. We rank-transform the target, feature, and controls; residualize feature and target against fixed controls; then compute the residual Spearman association. A rank-OLS coefficient with bootstrap sign consistency is reported as a second view.

In [ ]:
def rank_zscore(series: pd.Series) -> pd.Series:
    ranks = pd.to_numeric(series, errors="coerce").rank(method="average", pct=True)
    scale = ranks.std(ddof=0)
    return (ranks - ranks.mean()) / scale


def residualized_rank_check(frame: pd.DataFrame, feature: str, controls: list[str]) -> pd.DataFrame:
    clean = frame[[TARGET, feature, *controls]].replace([np.inf, -np.inf], np.nan).dropna()
    ranked = pd.DataFrame({column: rank_zscore(clean[column]) for column in clean.columns}).dropna()
    X = ranked[controls].to_numpy(dtype=float)
    y = ranked[TARGET].to_numpy(dtype=float)
    x = ranked[feature].to_numpy(dtype=float)

    y_resid = y - LinearRegression().fit(X, y).predict(X)
    x_resid = x - LinearRegression().fit(X, x).predict(X)
    residual_spearman = stats.spearmanr(x_resid, y_resid)

    design = np.column_stack([np.ones(len(ranked)), x, X])
    beta = float(np.linalg.lstsq(design, y, rcond=None)[0][1])
    rng = np.random.default_rng(RANDOM_SEED)
    boot = []
    for _ in range(2000):
        idx = rng.integers(0, len(ranked), len(ranked))
        boot.append(float(np.linalg.lstsq(design[idx], y[idx], rcond=None)[0][1]))
    boot = np.asarray(boot, dtype=float)

    return pd.DataFrame(
        [
            {
                "feature": feature,
                "n": len(ranked),
                "controls": ", ".join(controls),
                "partial_spearman_residual_rho": float(residual_spearman.statistic),
                "partial_spearman_p": float(residual_spearman.pvalue),
                "rank_ols_beta": beta,
                "rank_ols_ci_low": float(np.quantile(boot, 0.025)),
                "rank_ols_ci_high": float(np.quantile(boot, 0.975)),
                "rank_ols_sign_consistency": float(np.mean(np.sign(boot) == np.sign(beta))),
            }
        ]
    )


standard_general = load_matrix("standard_vs_foundational", "general")
controls = [
    column
    for column in ["log_n", "log_d", "d_over_n", "cat_fraction", "feature_missing_fraction"]
    if column in standard_general.columns
]
confound_check = residualized_rank_check(standard_general, "pymfe__attr_ent.skewness", controls)
display(confound_check)


## Low-Sample, High-p Diagnostics

The first-BH threshold is the absolute Spearman correlation needed for the most significant feature to pass the first Benjamini-Hochberg cutoff. The 80% power value is a Fisher-z approximation for the same per-feature alpha.

In [ ]:
def rho_for_two_sided_p(n: int, p_value: float) -> float:
    t_value = stats.t.isf(p_value / 2, n - 2)
    return float(math.sqrt(t_value**2 / (t_value**2 + n - 2)))


def rho_for_power(n: int, alpha: float, power: float = 0.80) -> float:
    z_value = (stats.norm.isf(alpha / 2) + stats.norm.isf(1 - power)) / math.sqrt(n - 3)
    return float(math.tanh(z_value))


def effective_rank(frame: pd.DataFrame) -> dict[str, float | int]:
    features = numeric_feature_columns(frame)
    X = frame[features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    values = SimpleImputer(strategy="median").fit_transform(X)
    scale = values.std(axis=0, ddof=0)
    keep = scale > 1e-12
    standardized = (values[:, keep] - values[:, keep].mean(axis=0)) / scale[keep]
    eigenvalues = np.clip(np.linalg.eigvalsh(np.corrcoef(standardized, rowvar=False)), 0, None)
    eigenvalues = eigenvalues[eigenvalues > 1e-12]
    probabilities = eigenvalues / eigenvalues.sum()
    return {
        "n": len(frame),
        "p": len(features),
        "rank_upper_bound": int(np.linalg.matrix_rank(standardized)),
        "participation_ratio_effective_rank": float(eigenvalues.sum() ** 2 / (eigenvalues**2).sum()),
        "entropy_effective_rank": float(np.exp(-(probabilities * np.log(probabilities)).sum())),
    }


regime_rows = []
for matrix in ["general", "classification"]:
    frame = load_matrix("standard_vs_foundational", matrix)
    row = effective_rank(frame)
    first_bh_alpha = 0.05 / row["p"]
    row.update(
        {
            "matrix": matrix,
            "first_bh_alpha": first_bh_alpha,
            "rho_first_bh_threshold": rho_for_two_sided_p(row["n"], first_bh_alpha),
            "rho_80_power_first_bh": rho_for_power(row["n"], first_bh_alpha),
            "rho_80_power_uncorrected_0_05": rho_for_power(row["n"], 0.05),
        }
    )
    regime_rows.append(row)

regime_diagnostics = pd.DataFrame(regime_rows)
display(regime_diagnostics)


## Simple LODO Predictive Baselines

These baselines answer whether the predictive result is only failing because the all-feature model is too diffuse. The nested top-one model selects the strongest univariate predictor using only the training datasets in each leave-one-dataset-out fold. These simple diagnostics are secondary and hypothesis-generating.

In [ ]:
def ridge_lodo_metrics(frame: pd.DataFrame, predictors: list[str], *, model: str) -> dict[str, object]:
    y_true = frame[TARGET].astype(float).to_numpy()
    predictions = []
    for test_idx in range(len(frame)):
        train = frame.drop(frame.index[test_idx])
        test = frame.iloc[[test_idx]]
        if not predictors:
            predictions.append(float(train[TARGET].mean()))
            continue
        X_train = train[predictors].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        X_test = test[predictors].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        estimator = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
                ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 13), cv=None)),
            ]
        )
        estimator.fit(X_train, train[TARGET])
        predictions.append(float(estimator.predict(X_test)[0]))
    predictions = np.asarray(predictions, dtype=float)
    sse = float(np.sum((y_true - predictions) ** 2))
    sst = float(np.sum((y_true - y_true.mean()) ** 2))
    return {
        "model": model,
        "predictors": ", ".join(predictors),
        "mae": float(mean_absolute_error(y_true, predictions)),
        "rmse": float(mean_squared_error(y_true, predictions) ** 0.5),
        "oos_r2": float(1 - sse / sst),
        "spearman_pred_obs": float(stats.spearmanr(y_true, predictions).statistic) if np.unique(predictions).size > 1 else np.nan,
        "sign_accuracy": float(np.mean(np.sign(y_true[y_true != 0]) == np.sign(predictions[y_true != 0]))),
    }


def nested_top_one_lodo(frame: pd.DataFrame) -> dict[str, object]:
    features = numeric_feature_columns(frame)
    y_true = frame[TARGET].astype(float).to_numpy()
    predictions = []
    selected_features = []
    for test_idx in range(len(frame)):
        train = frame.drop(frame.index[test_idx])
        test = frame.iloc[[test_idx]]
        X = train[features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        y_ranked = train[TARGET].rank(method="average")
        correlations = X.rank(method="average").corrwith(y_ranked, method="pearson").abs().dropna()
        correlations = correlations[(X.notna().sum() >= MIN_N) & (X.nunique(dropna=True) > 1)]
        if correlations.empty:
            predictions.append(float(train[TARGET].mean()))
            continue
        selected = correlations.sort_values(ascending=False).index[0]
        selected_features.append(selected)
        # Refit only for this fold so the selected feature is training-fold specific.
        X_train = pd.to_numeric(train[selected], errors="coerce").to_frame().replace([np.inf, -np.inf], np.nan)
        X_test = pd.to_numeric(test[selected], errors="coerce").to_frame().replace([np.inf, -np.inf], np.nan)
        estimator = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
                ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 13), cv=None)),
            ]
        )
        estimator.fit(X_train, train[TARGET])
        predictions.append(float(estimator.predict(X_test)[0]))
    predictions = np.asarray(predictions, dtype=float)
    sse = float(np.sum((y_true - predictions) ** 2))
    sst = float(np.sum((y_true - y_true.mean()) ** 2))
    return {
        "model": "ridge_nested_top1",
        "predictors": json.dumps(pd.Series(selected_features).value_counts().head(5).to_dict()),
        "mae": float(mean_absolute_error(y_true, predictions)),
        "rmse": float(mean_squared_error(y_true, predictions) ** 0.5),
        "oos_r2": float(1 - sse / sst),
        "spearman_pred_obs": float(stats.spearmanr(y_true, predictions).statistic),
        "sign_accuracy": float(np.mean(np.sign(y_true[y_true != 0]) == np.sign(predictions[y_true != 0]))),
    }


def simple_lodo_baselines(frame: pd.DataFrame) -> pd.DataFrame:
    controls = [
        column
        for column in ["log_n", "log_d", "d_over_n", "cat_fraction", "feature_missing_fraction", "n_classes", "class_imbalance_ratio"]
        if column in frame.columns
    ]
    rows = [ridge_lodo_metrics(frame, [], model="mean_baseline")]
    if "log_n" in frame.columns:
        rows.append(ridge_lodo_metrics(frame, ["log_n"], model="ridge_log_n"))
    if controls:
        rows.append(ridge_lodo_metrics(frame, controls, model="ridge_basic_controls"))
    rows.append(nested_top_one_lodo(frame))
    output = pd.DataFrame(rows)
    output["delta_mae_vs_mean"] = output.loc[output["model"].eq("mean_baseline"), "mae"].iloc[0] - output["mae"]
    return output


for comparison in CONFIG_HANDOFFS:
    for matrix in ["general", "classification"]:
        print(f"{comparison} / {matrix}")
        display(simple_lodo_baselines(load_matrix(comparison, matrix)))
